# Train Tokenizer

- using `rustbpe` tokenizer, equivalent to HF but simpler
- this should **exactly** match tokenizer produced by NanoChat

In [56]:
import datasets
import rustbpe
import tiktoken
import pickle

In [57]:
dataset = datasets.load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-100BT",
    split="train",
)
dataset = dataset.shuffle(seed=42)  # Match nanochat repackage_data_reference.py seed

In [58]:
# For Testing
# max_chars = 2_000_000
# vocab_size = 1024
# doc_cap = 10000

# Match params used in nanochat speedrun.sh
# python -m scripts.tok_train --max_chars=2000000000 --vocab_size=65536
max_chars = 2000000000
vocab_size = 65536
doc_cap = 10000

# from nanochat tokenizer.py
# NOTE: this split pattern deviates from GPT-4 in that we use \p{N}{1,2} instead of \p{N}{1,3}
# I did this because I didn't want to "waste" too many tokens on numbers for smaller vocab sizes.
# I haven't validated that this is actually a good idea
SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,2}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

# from nanochat tokenizer.py
SPECIAL_TOKENS = [
    # every document begins with the Beginning of Sequence (BOS) token that delimits documents
    "<|bos|>",
    # tokens below are only used during finetuning to render Conversations into token ids
    "<|user_start|>", # user messages
    "<|user_end|>",
    "<|assistant_start|>", # assistant messages
    "<|assistant_end|>",
    "<|python_start|>", # assistant invokes python REPL tool
    "<|python_end|>",
    "<|output_start|>", # python REPL outputs back to assistant
    "<|output_end|>",
]

In [60]:
train_docs = []
char_count = 0
for i, example in enumerate(dataset):
    text = example["text"]
    if len(text) > doc_cap:
        text = text[:doc_cap]
    train_docs.append(text)
    char_count += len(text)
    
    if i % 100000 == 0 or char_count >= max_chars:
        pct = (char_count / max_chars) * 100
        print(f"Processed {char_count} / {max_chars} ({pct:.2f}%)")
    
    if char_count >= max_chars:
        break

Processed 8657 / 2000000000 (0.00%)
Processed 376164670 / 2000000000 (18.81%)
Processed 751674822 / 2000000000 (37.58%)
Processed 1126477659 / 2000000000 (56.32%)
Processed 1502706986 / 2000000000 (75.14%)
Processed 1877824553 / 2000000000 (93.89%)
Processed 2000002067 / 2000000000 (100.00%)


In [61]:
# Create tokenizer and train on your data
vocab_size_no_specials = vocab_size - len(SPECIAL_TOKENS)
tokenizer = rustbpe.Tokenizer()
tokenizer.train_from_iterator(
    train_docs,
    vocab_size=vocab_size_no_specials,
    pattern=SPLIT_PATTERN
)

In [62]:
pattern = tokenizer.get_pattern()
mergeable_ranks_list = tokenizer.get_mergeable_ranks()
mergeable_ranks = {bytes(k): v for k, v in mergeable_ranks_list}
tokens_offset = len(mergeable_ranks)
special_tokens = {name: tokens_offset + i for i, name in enumerate(SPECIAL_TOKENS)}
enc = tiktoken.Encoding(
    name="rustbpe",
    pat_str=pattern,
    mergeable_ranks=mergeable_ranks, # dict[bytes, int] (token bytes -> merge priority rank)
    special_tokens=special_tokens, # dict[str, int] (special token name -> token id)
)

In [63]:
with open("../data/tokenizer.pkl", "wb") as f:
    pickle.dump(enc, f)

In [ ]:
token_strings = [enc.decode([i]) for i in range(enc.n_vocab)]  # list[str] (token id -> token string)
token_bytes = []
for i in range(enc.n_vocab):
    tok_str = token_strings[i]
    if tok_str in special_tokens:
        # special tokens are not byte sequences
        token_bytes.append(0)
    else:
        token_bytes.append(len(tok_str.encode("utf-8")))

with open("../data/token_bytes.pkl", "wb") as f:
    pickle.dump(token_bytes, f)

# Validate against NanoChat

In [65]:
enc2 = pickle.load(open("/home/user/.cache/nanochat/tokenizer/tokenizer.pkl", "rb"))

In [71]:
print(enc2._pat_str)
enc2._pat_str == enc._pat_str

'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,2}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+


True

In [72]:
print(len(enc2._special_tokens))
enc2._special_tokens == enc._special_tokens

9


True

In [73]:
print(len(enc2._mergeable_ranks), len(enc._mergeable_ranks))
enc2._mergeable_ranks == enc._mergeable_ranks

65527 65527


True

In [74]:
import torch
token_bytes2 = torch.load("/home/user/.cache/nanochat/tokenizer/token_bytes.pt")
token_bytes2 = token_bytes2.tolist()

In [76]:
print(len(token_bytes2))
token_bytes2 == token_bytes

65536


True

In [77]:
for i, example in enumerate(dataset):
    tokens = enc.encode_ordinary(example["text"])
    tokens2 = enc2.encode_ordinary(example["text"])
    assert tokens2 == tokens
    if i > 1000:
        break
print(f"Checked tokenization consistency on {i+1} examples successfully.")

Checked tokenization consistency on 1002 examples successfully.
